# MODELO PREDICTIVO

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LogisticRegression
import os


In [2]:
df = pd.read_csv(r'C:\Users\lucia\Documents\Master\TFM\marketing_analysis_fintech\datos\processed_data\FINTECH_LIMPIO.csv', sep=',')

In [4]:
df['subscribed'].value_counts()

subscribed
no     35654
yes     4532
Name: count, dtype: int64

In [29]:
import numpy as np
import pandas as pd
import os
import shutil
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, 
    roc_curve, 
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score
)

def calcula_metricas(metricas, y_test, y_pred, y_prob, fold):
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    metricas.append({
        "fold": fold,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auc": auc
    })


def viz_metricas(y_test, y_pred, y_prob, fold, resultados_dir):
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))

    # MATRIZ DE CONFUSION
    cm = confusion_matrix(y_test, y_pred)
    im = ax[0].imshow(cm, cmap="Blues")
    # Títulos y ejes
    ax[0].set_title("Matriz de confusión")
    ax[0].set_xlabel("Predicho")
    ax[0].set_ylabel("Real")
    # Etiquetas (ajusta según tu problema)
    labels = ["No", "Yes"]
    ax[0].set_xticks(np.arange(len(labels)))
    ax[0].set_yticks(np.arange(len(labels)))
    ax[0].set_xticklabels(labels)
    ax[0].set_yticklabels(labels)
    # Añadir números dentro de cada celda
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax[0].text(j, i, cm[i, j],
                    ha="center", va="center",
                    color="black", fontsize=12)
    # Barra de color
    fig.colorbar(im, ax=ax[0])

    # curva roc
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax[1].plot(fpr, tpr)
    ax[1].plot([0,1],[0,1])
    ax[1].set_title("Curva ROC")
    ax[1].set_xlabel("Ratio falso positivo")
    ax[1].set_ylabel("Ratio verdadero positivo")

    # 🔹 Guardar figura
    filename = os.path.join(resultados_dir, f"metricas_fold_{fold}.png")
    plt.savefig(filename)

    plt.close()  # muy importante para no acumular memoria

def flujo_kfold(y, X, kf, model, resultados_dir):
    metricas = []
    for fold, (train_index, test_index) in enumerate(kf.split(X), start=1):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:,1]

        calcula_metricas(metricas, y_test, y_pred, y_prob, fold)

        viz_metricas(y_test, y_pred, y_prob, fold, resultados_dir)

        # tabla de metricas
        df_metricas = pd.DataFrame(metricas)

        print("\nMetricas por fold")
        print(df_metricas)

        print("\nMedia de metricas")
        print(df_metricas.mean())


In [30]:
resultados_dir = "resultados_modelo"
if os.path.exists(resultados_dir):
    shutil.rmtree(resultados_dir)  
os.makedirs(resultados_dir)   

cols = df.columns.drop('subscribed')

y = df['subscribed'].map({'no':0, 'yes':1})
X = pd.get_dummies(df[cols])

kf = KFold(n_splits=10, shuffle=True, random_state=42)

model = LogisticRegression(max_iter=200)

flujo_kfold(y, X, kf, model, resultados_dir)

c:\Users\lucia\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Metricas por fold
   fold  accuracy  precision    recall       f1       auc
0     1  0.908684   0.654804  0.405286  0.50068  0.930645

Media de metricas
fold         1.000000
accuracy     0.908684
precision    0.654804
recall       0.405286
f1           0.500680
auc          0.930645
dtype: float64


c:\Users\lucia\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Metricas por fold
   fold  accuracy  precision    recall        f1       auc
0     1  0.908684   0.654804  0.405286  0.500680  0.930645
1     2  0.908684   0.660448  0.390728  0.490985  0.931990

Media de metricas
fold         1.500000
accuracy     0.908684
precision    0.657626
recall       0.398007
f1           0.495833
auc          0.931317
dtype: float64


c:\Users\lucia\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


...


Metricas por fold
   fold  accuracy  precision    recall        f1       auc
0     1  0.908684   0.654804  0.405286  0.500680  0.930645
1     2  0.908684   0.660448  0.390728  0.490985  0.931990
2     3  0.913660   0.585443  0.461347  0.516039  0.929925
3     4  0.909181   0.703226  0.443992  0.544320  0.934998
4     5  0.915651   0.693431  0.426966  0.528512  0.929745
5     6  0.901468   0.644444  0.416838  0.506234  0.930595
6     7  0.910901   0.666667  0.429825  0.522667  0.924308
7     8  0.908163   0.655405  0.420824  0.512550  0.924052

Media de metricas
fold         4.500000
accuracy     0.909549
precision    0.657984
recall       0.424476
f1           0.515248
auc          0.929532
dtype: float64


c:\Users\lucia\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Metricas por fold
   fold  accuracy  precision    recall        f1       auc
0     1  0.908684   0.654804  0.405286  0.500680  0.930645
1     2  0.908684   0.660448  0.390728  0.490985  0.931990
2     3  0.913660   0.585443  0.461347  0.516039  0.929925
3     4  0.909181   0.703226  0.443992  0.544320  0.934998
4     5  0.915651   0.693431  0.426966  0.528512  0.929745
5     6  0.901468   0.644444  0.416838  0.506234  0.930595
6     7  0.910901   0.666667  0.429825  0.522667  0.924308
7     8  0.908163   0.655405  0.420824  0.512550  0.924052
8     9  0.913639   0.670290  0.419501  0.516039  0.936876

Media de metricas
fold         5.000000
accuracy     0.910003
precision    0.659351
recall       0.423923
f1           0.515336
auc          0.930348
dtype: float64


c:\Users\lucia\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Metricas por fold
   fold  accuracy  precision    recall        f1       auc
0     1  0.908684   0.654804  0.405286  0.500680  0.930645
1     2  0.908684   0.660448  0.390728  0.490985  0.931990
2     3  0.913660   0.585443  0.461347  0.516039  0.929925
3     4  0.909181   0.703226  0.443992  0.544320  0.934998
4     5  0.915651   0.693431  0.426966  0.528512  0.929745
5     6  0.901468   0.644444  0.416838  0.506234  0.930595
6     7  0.910901   0.666667  0.429825  0.522667  0.924308
7     8  0.908163   0.655405  0.420824  0.512550  0.924052
8     9  0.913639   0.670290  0.419501  0.516039  0.936876
9    10  0.913390   0.671480  0.419865  0.516667  0.931549

Media de metricas
fold         5.500000
accuracy     0.910342
precision    0.660564
recall       0.423517
f1           0.515469
auc          0.930468
dtype: float64


In [31]:
resultados_dir = "resultados_modelo_sample"
if os.path.exists(resultados_dir):
    shutil.rmtree(resultados_dir)  
os.makedirs(resultados_dir)   

df_no = df[df['subscribed'] == 'no']
df_si = df[df['subscribed'] == 'yes']
df_no_sample = df_no.sample(n=7000, random_state=1)
df_sample = pd.concat([df_no_sample, df_si])

cols = df_sample.columns.drop('subscribed')

y = df_sample['subscribed'].map({'no':0, 'yes':1})
X = pd.get_dummies(df_sample[cols])

kf = KFold(n_splits=10, shuffle=True, random_state=42)

model = LogisticRegression(max_iter=200)

flujo_kfold(y, X, kf, model, resultados_dir)

c:\Users\lucia\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Metricas por fold
   fold  accuracy  precision    recall        f1       auc
0     1  0.844887   0.800459  0.791383  0.795895  0.929928

Media de metricas
fold         1.000000
accuracy     0.844887
precision    0.800459
recall       0.791383
f1           0.795895
auc          0.929928
dtype: float64


c:\Users\lucia\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Metricas por fold
   fold  accuracy  precision    recall        f1       auc
0     1  0.844887   0.800459  0.791383  0.795895  0.929928
1     2  0.851820   0.798673  0.818594  0.808511  0.929629

Media de metricas
fold         1.500000
accuracy     0.848354
precision    0.799566
recall       0.804989
f1           0.802203
auc          0.929778
dtype: float64


c:\Users\lucia\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


...


Metricas por fold
   fold  accuracy  precision    recall        f1       auc
0     1  0.844887   0.800459  0.791383  0.795895  0.929928
1     2  0.851820   0.798673  0.818594  0.808511  0.929629
2     3  0.856028   0.842105  0.778761  0.809195  0.942165
3     4  0.838682   0.807425  0.771619  0.789116  0.923832
4     5  0.869037   0.825503  0.834842  0.830146  0.938691
5     6  0.857762   0.837587  0.793407  0.814898  0.934847
6     7  0.855160   0.823666  0.795964  0.809578  0.927747
7     8  0.849089   0.818947  0.815514  0.817227  0.930036

Media de metricas
fold         4.500000
accuracy     0.852808
precision    0.819296
recall       0.800010
f1           0.809321
auc          0.932109
dtype: float64


c:\Users\lucia\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Metricas por fold
   fold  accuracy  precision    recall        f1       auc
0     1  0.844887   0.800459  0.791383  0.795895  0.929928
1     2  0.851820   0.798673  0.818594  0.808511  0.929629
2     3  0.856028   0.842105  0.778761  0.809195  0.942165
3     4  0.838682   0.807425  0.771619  0.789116  0.923832
4     5  0.869037   0.825503  0.834842  0.830146  0.938691
5     6  0.857762   0.837587  0.793407  0.814898  0.934847
6     7  0.855160   0.823666  0.795964  0.809578  0.927747
7     8  0.849089   0.818947  0.815514  0.817227  0.930036
8     9  0.872507   0.849372  0.844075  0.846715  0.945237

Media de metricas
fold         5.000000
accuracy     0.854997
precision    0.822637
recall       0.804906
f1           0.813476
auc          0.933568
dtype: float64


c:\Users\lucia\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Metricas por fold
   fold  accuracy  precision    recall        f1       auc
0     1  0.844887   0.800459  0.791383  0.795895  0.929928
1     2  0.851820   0.798673  0.818594  0.808511  0.929629
2     3  0.856028   0.842105  0.778761  0.809195  0.942165
3     4  0.838682   0.807425  0.771619  0.789116  0.923832
4     5  0.869037   0.825503  0.834842  0.830146  0.938691
5     6  0.857762   0.837587  0.793407  0.814898  0.934847
6     7  0.855160   0.823666  0.795964  0.809578  0.927747
7     8  0.849089   0.818947  0.815514  0.817227  0.930036
8     9  0.872507   0.849372  0.844075  0.846715  0.945237
9    10  0.849089   0.810502  0.795964  0.803167  0.928806

Media de metricas
fold         5.500000
accuracy     0.854406
precision    0.821424
recall       0.804012
f1           0.812445
auc          0.933092
dtype: float64
